# Mamba SOH — NOWCASTING multi-battery + leave-one-battery-out (GH-13)

Trả lời câu hỏi: **thêm pin (4 → 33) có giúp model nowcasting không**, và đo **MAE ± std** đáng tin qua leave-one-battery-out trên GPU.

**Trước khi chạy:**
1. Settings → Accelerator → **GPU P100/T4**
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. (repo private) Add-ons → Secrets → tạo `GITHUB_TOKEN` = GitHub PAT

> Notebook KHÔNG sửa file — chạy thẳng các experiment script từ branch GH-13. Model production window=30 (4 pin) giữ nguyên.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: bật GPU ở Settings -> Accelerator -> GPU P100/T4')

## 2 — Clone branch `feat/spectral_kurtosis` (đã merge GH-13)

In [ ]:
import subprocess
# GH-13 đã merge (PR #14) vào feat/spectral_kurtosis → clone nhánh này để có đủ experiment scripts
BRANCH  = 'feat/spectral_kurtosis'
REPO    = '/kaggle/working/ai-module'
URL_PUB = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thử public clone:', e)
    url = URL_PUB
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', url, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL_PUB])
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git','-C',REPO,'log','-1','--oneline']).decode().strip())

## 3 — Dependencies

In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 4 — Tìm NASA dataset + vào repo

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv — + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET    :', DATASET)
print('has data/  :', os.path.isdir(f'{DATASET}/data'))
print('cwd        :', os.getcwd())

## 5 — Multi-battery nowcasting (train 33 pin, test B0018)

Trả lời: thêm pin có giúp nowcasting trên B0018 không (so baseline 4-pin 0.61%).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/experiment_nowcast_multi.py --data-dir "{DATASET}" --epochs 80

## 6 — Leave-one-battery-out (con số ĐÁNG TIN: MAE ± std)

Xoay vòng giữ mỗi pin làm test → báo cáo MAE/RMSE trung bình ± độ lệch qua tất cả pin.
Đây là con số bảo vệ trước hội đồng, không phải 1 điểm từ 1 pin.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/experiment_nowcast_lobo.py --data-dir "{DATASET}" --epochs 60 --min-cycles 60

## 7 — Kết quả

- **Cell 5** cho biết thêm pin GIÚP / comparable / HẠI trên B0018.
- **Cell 6** cho con số `MAE ± std` qua nhiều pin — bằng chứng độ tin cậy.
- Log đầy đủ ở `logs/training/train_*.log` (tab Output).

> Nếu LOBO quá chậm: giảm `--epochs` hoặc giới hạn fold bằng `--folds B0005,B0006,B0007,B0018,B0033,B0042`.